**Install Required Packages**

In [ ]:
!pip install pandas --quiet
!pip install tabulate --quiet
!pip install pdfplumber --quiet
!pip install pymupdf --upgrade --prefer-binary --only-binary :all:

**Define page-finder & table-extractor functions**

In [ ]:
import pandas as pd

def find_pages(pdf_path: str, keyword: str) -> list[int]:
    """
    Return 1-based page numbers where `keyword` appears in the page text.
    """
    pages = []
    with fitz.open(pdf_path) as doc:
        for i in range(doc.page_count):
            if keyword in doc[i].get_text():
                pages.append(i + 1)
    return pages


def extract_tables(
    pdf_path: str,
    pages: list[int],
    period: str,
    drop_type: str = "Spread") -> pd.DataFrame:
    """
    From the given pages, pull out every table row where
    df['Period']==period AND df['Type']!=drop_type.
    Returns one concatenated DataFrame (or empty if none).
    """
    dfs = []
    with pdfplumber.open(pdf_path) as pdf:
        for pnum in pages:
            t = pdf.pages[pnum - 1].extract_table()
            if not t:
                continue
            df = pd.DataFrame(t[1:], columns=t[0])
            if "Type" in df.columns:
                df["Type"] = df["Type"].str.strip()
                df = df[df["Type"] != drop_type]
            if "Period" in df.columns:
                df = df[df["Period"] == period]
            if not df.empty:
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

**Define a function to determine if "In the Money"**

In [ ]:
def find_in_the_money(df: pd.DataFrame) -> pd.DataFrame:
    """
    Coerce 'Exp Value' to numeric, extract numeric threshold from 'Display Name',
    compute 'Flag' as 1 if Exp Value > Threshold (else 0), and drop the intermediate 'Threshold' column.
    """
    return (
        df
        .assign(
            **{'Exp Value': lambda df: pd.to_numeric(df['Exp Value'], errors='coerce')},
            Threshold=lambda df: (
                df['Display Name']
                  .str.extract(r'>\s*([\d.]+)', expand=False)
                  .pipe(pd.to_numeric, errors='coerce')
            ),
        )
        .assign(
            Flag=lambda df: (df['Exp Value'] > df['Threshold']).astype(int)
        )
        .drop(columns=['Threshold'])
    )


**Define the clean and rename function**

In [ ]:
import pandas as pd

def clean_and_rename(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Keeps the first unique 'Display Name'
    - Drops columns: Name, Period, Display Name, Type, Buyer, Seller, Exp Time
    - Renames:
        Business Date  → Date
        Exp Value      → Strike
        Flag           → In the Money
    """
    return (
        df
        .drop_duplicates(subset=["Display Name"], keep="first")
        .drop(columns=["Type", "Buyer", "Seller", "Exp Time"])
        .drop(columns=["Period", "Display Name"])
        .rename(columns={
            "Business Date": "Date",
            "Exp Value": "Strike",
            "Flag": "In the Money",
        })
    )

**Define a function to substitue the Ticker for the name**

In [ ]:
import pandas as pd
from pathlib import Path
from typing import Union

def add_ticker_from_mapping(
    df: pd.DataFrame,
    mapping_file: Union[str, Path]) -> pd.DataFrame:
    """
    Load a mapping CSV with columns 'Display Name' and 'Ticker',
    then add a 'Ticker' column to `df` by substring-matching 'Name' against each 'Display Name'.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame containing a 'Name' column.
    mapping_file : str or Path
        Path to CSV with mapping of 'Display Name' to 'Ticker'.

    Returns
    -------
    pd.DataFrame
        A new DataFrame with an added 'Ticker' column.
    """
    mapping_df = pd.read_csv(mapping_file)
    
    def lookup_ticker(name: str) -> pd.Series:
        if pd.isna(name):
            return pd.NA
        for desc, ticker in zip(mapping_df['Display Name'], mapping_df['Ticker']):
            if desc in name:
                return ticker
        return pd.NA

    result = df.copy()
    result['Ticker'] = result['Name'].apply(lookup_ticker)
    return result

# Example usage:
# cleaned_df = add_ticker_from_mapping(result, Path('.') / "ticker_from_description.csv")


**Define a function to save the final result to an S3 CSV**

In [ ]:
import io
import boto3
import pandas as pd
from botocore.exceptions import ClientError

def upload_df_to_s3(
    df: pd.DataFrame,
    bucket: str,
    key: str,
    region: str = None) -> None:
    """
    Uploads a DataFrame to S3 as CSV. Verifies bucket existence first.
    
    Parameters
    ----------
    df : pd.DataFrame
    bucket : str
        Name of the S3 bucket (no leading/trailing spaces).
    key : str
        S3 object key, e.g. "csv/2025-06-28-results.csv"
    region : str, optional
        AWS region where the bucket resides.
    """
    bucket = bucket.strip()
    s3 = boto3.client('s3', region_name=region)
    
    # 1) Check bucket
    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError as e:
        code = e.response['Error']['Code']
        msg  = e.response['Error']['Message']
        raise RuntimeError(
            f"Could not access bucket '{bucket}' (region={region}): {msg} (code {code})"
        ) from e

    # 2) Serialize and upload
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)

    try:
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
        print(f"✅ Uploaded to s3://{bucket}/{key}")
    except ClientError as e:
        raise RuntimeError(
            f"Failed to upload CSV to s3://{bucket}/{key}: "
            f"{e.response['Error']['Message']}"
        ) from e


**Define function to avoid re-processing processed files**

In [ ]:
import json
import boto3

# S3 client and bucket name
s3 = boto3.client('s3', region_name='us-east-1')
bucket = 'nadex-daily-results'

from botocore.exceptions import ClientError

def load_manifest() -> set:
    """
    Download and parse the JSON manifest of processed PDFs using the client.
    Returns a set of keys, or empty set if the object doesn’t exist.
    """
    try:
        resp = s3_client.get_object(
            Bucket=bucket_name,
            Key="manifests/processed_files.json"
        )
        return set(json.loads(resp["Body"].read()))
    except ClientError as e:
        # If it’s not found, return empty; re-raise other errors
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return set()
        raise

def save_manifest(processed: set):
    """
    Write the updated manifest back to S3 via the client.
    """
    s3_client.put_object(
        Bucket=bucket_name,
        Key="manifests/processed_files.json",
        Body=json.dumps(list(processed))
    )

def list_s3_keys(prefix: str) -> list:
    """
    List all object keys under the given prefix, handling pagination.
    """
    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            keys.append(obj['Key'])
    return keys

**Define a function to get all the files related to trading results**

In [ ]:
def list_nadex_trading_results(prefix: str = "") -> list[str]:
    return [
        obj.key
        for obj in nadex_bucket.objects.filter(Prefix=prefix)
        if "tradingResults" in obj.key and obj.key.lower().endswith(".pdf")
    ]

**Load the manifest, and find new files**

In [ ]:
import boto3
from botocore.client import Config
from botocore import UNSIGNED
from pathlib import Path

# 1) Bootstrap a single Session, client, and resource
session      = boto3.Session(profile_name="default", region_name="us-east-1")
public_s3 = session.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
    region_name="us-east-1"   # region still required
)
s3_client    = session.client("s3")
s3_resource  = session.resource("s3")

# 2) Wire up your buckets
bucket_name       = "nadex-daily-results"
bucket            = s3_resource.Bucket(bucket_name)

nadex_bucket_name = "market-data-prod.nadex.com"
nadex_bucket      = s3_resource.Bucket(nadex_bucket_name)

# 3) Use it in your main block
processed = load_manifest()
all_nadex = list_nadex_trading_results()      
new_pdfs  = [k for k in all_nadex if k not in processed]

**Run the pipeline & display**

1. For each PDF in the folder:
2. Read the PDF file
3. Extract the Tables with Daily contracts
4. Create a CSV
5. Write the CSV to S3
6. Log the PDF file as 'processed

In [ ]:
from pathlib import Path

import pandas as pd
import fitz
import pdfplumber

from botocore import UNSIGNED
from botocore.client import Config

from tabulate import tabulate

import traceback

# ─── Definitions ───────────────────────────────────────────────────────────────

pdf_folder   = Path('.')  
mapping_file = pdf_folder / "ticker_from_description.csv"
target       = "Daily"

public_s3 = session.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
    region_name="us-east-1"
)

errors = []

# ─── your loop ────────────────────────────────────────────────────────────────
try:
    for pdf_key in new_pdfs:
        fname     = Path(pdf_key).name
        local_pdf = Path("/tmp") / fname

        try:
            # 1) Download (public Nadex bucket, unsigned client)
            public_s3.download_file(
                Bucket = nadex_bucket_name,
                Key    = pdf_key,
                Filename = str(local_pdf)
            )

            # 2) Extract & transform
            pages    = find_pages(str(local_pdf), target)
            final_df = (
                extract_tables(str(local_pdf), pages, target)
                .pipe(find_in_the_money)
                .pipe(clean_and_rename)
                .pipe(add_ticker_from_mapping, mapping_file)
            )

            # 3) Preview
            # print(tabulate(final_df.tail(5), headers="keys", tablefmt="fancy_grid", showindex=False))

            # 4) Upload result CSV
            s3_key = f"csv/{Path(pdf_key).stem}.csv"
            upload_df_to_s3(final_df, bucket=bucket_name, key=s3_key, region="us-east-1")

            # 5) Mark as processed
            processed.add(pdf_key)

        except Exception as e:
            # Log the error and move on to the next file
            print(f"⚠️ Error processing {pdf_key}: {e}")
            errors.append(pdf_key)
            continue

finally:
    save_manifest(processed)
    print(f"\n✅ Manifest saved. {len(processed)} total files marked processed.")
    if errors:
        print(f"❗ Failed on {len(errors)} files:", errors)